In [4]:
import pyarrow.parquet as pq
import pandas as pd
import json
from collections import Counter

# Đọc file parquet
table = pq.read_table("train-00001-of-00002.parquet")
df = table.to_pandas()

print("Columns:", df.columns.tolist())

# =========================
# Parse cột meta
# =========================

def parse_meta(x):
    """
    Chuyển meta về dict.
    Hỗ trợ:
    - dict
    - string JSON
    """
    if isinstance(x, dict):
        return x

    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return {}

    return {}

meta_parsed = df["meta"].apply(parse_meta)

# =========================
# Thống kê issuing_agency
# =========================

issuing_agency_counter = Counter()

for meta in meta_parsed:
    agency = meta.get("issuing_agency")

    if agency is not None:
        issuing_agency_counter[agency] += 1

print("\n===== THỐNG KÊ issuing_agency =====")
print(f"Số loại khác nhau: {len(issuing_agency_counter)}\n")

for agency, count in issuing_agency_counter.most_common():
    print(f"{agency}: {count}")


# =========================
# Thống kê type
# =========================

type_counter = Counter()

for meta in meta_parsed:
    t = meta.get("type")

    if t is not None:
        type_counter[t] += 1

print("\n===== THỐNG KÊ type =====")
print(f"Số loại khác nhau: {len(type_counter)}\n")

for t, count in type_counter.most_common():
    print(f"{t}: {count}")

Columns: ['text', 'meta', 'content', 'citation', 'qas', 'task_type']

===== THỐNG KÊ issuing_agency =====
Số loại khác nhau: 32

Chính phủ: 68495
Bộ Tài chính: 6991
Bộ Giao thông vận tải: 3796
Bộ Nông nghiệp và Phát triển nông thôn: 2290
Bộ Y tế: 2199
Bộ Công thương: 2187
Ngân hàng Nhà nước: 1981
Bộ Tài nguyên và Môi trường: 1663
Bộ Quốc phòng: 1533
Bộ Công An: 1449
Bộ Khoa học và Công nghệ: 1398
Bộ Thông tin và Truyền thông: 1132
Bộ Lao động - Thương binh và Xã hội: 1059
Bộ Giáo dục và Đào tạo: 975
Ngân hàng Nhà nước Việt Nam: 904
Bộ Văn hoá, Thể thao và du lịch: 887
Bộ Xây dựng: 789
Bộ Tư pháp: 755
Bộ Kế hoạch và Đầu tư: 504
Bộ Lao động – Thương binh và Xã hội: 407
Bộ Nội vụ: 370
Thanh tra Chính phủ: 319
Uỷ ban Dân tộc: 227
Bộ Ngoại giao: 97
Bộ Tài nguyên môi trường: 94
Văn phòng Chính phủ: 64
Tòa án nhân dân tối cao: 59
Bộ Lao động thương binh và Xã hội: 48
Bộ Lao động- Thương binh và xã hội: 4
Ngân hàng Nông nghiệp Việt nam: 3
Tổng cục Địa chính: 3
Viện kiểm sát nhân dân tối cao: 2

In [1]:
import pyarrow.parquet as pq
import pandas as pd
import chromadb
import json

# =========================
# Đọc parquet (Giữ nguyên)
# =========================
table = pq.read_table("train-00001-of-00002.parquet")
df = table.to_pandas()
print("Số dòng ban đầu:", len(df))

# =========================
# Khởi tạo ChromaDB
# =========================
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="legal_documents")

# =========================
# Tối ưu hóa xử lý dữ liệu (Bỏ hoàn toàn iterrows)
# =========================

# 1. Lọc nhanh các dòng không có text hợp lệ
df = df[df["text"].notna() & (df["text"].astype(str).str.strip() != "")]

# 2. Hàm parse và ép kiểu metadata tối ưu hơn
def clean_metadata(x):
    # Parse JSON nếu là chuỗi
    if isinstance(x, str):
        try:
            meta = json.loads(x)
        except Exception:
            return {}
    elif isinstance(x, dict):
        meta = x
    else:
        return {}
    
    # Ép kiểu các giá trị không phải primitive về string cho ChromaDB
    return {
        k: v if isinstance(v, (str, int, float, bool)) else str(v) 
        for k, v in meta.items()
    }

# Sử dụng .apply() thay vì loop - nhanh hơn iterrows rất nhiều
print("Đang xử lý metadata...")
df["clean_meta"] = df["meta"].apply(clean_metadata)

# Tạo mảng IDs bằng list comprehension (nhanh và tốn ít bộ nhớ)
df["doc_id"] = [f"doc_{idx}" for idx in df.index]

# Chuyển đổi sang list để chuẩn bị insert
all_documents = df["text"].astype(str).tolist()
all_metadatas = df["clean_meta"].tolist()
all_ids = df["doc_id"].tolist()

print("Số dòng sau khi lọc:", len(all_documents))

# =========================
# Insert theo batch
# =========================
BATCH_SIZE = 1000

print("Đang insert vào ChromaDB...")
for i in range(0, len(all_documents), BATCH_SIZE):
    collection.add(
        documents=all_documents[i : i + BATCH_SIZE],
        metadatas=all_metadatas[i : i + BATCH_SIZE],
        ids=all_ids[i : i + BATCH_SIZE]
    )

print("Đã insert vào ChromaDB")
print("Tổng documents hiện tại:", collection.count())

Số dòng ban đầu: 102684
Đang xử lý metadata...
Số dòng sau khi lọc: 102684
Đang insert vào ChromaDB...
Đã insert vào ChromaDB
Tổng documents hiện tại: 102684


In [ ]:
import json
import pandas as pd
import pyarrow.parquet as pq
import chromadb
from tqdm import tqdm

# =========================
# Kết nối ChromaDB cũ
# =========================

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="legal_documents"
)

print("Documents hiện tại:", collection.count())

# =========================
# Đọc file mới
# =========================

table = pq.read_table("new_file.parquet")
df = table.to_pandas()

# =========================
# Parse metadata
# =========================

def parse_meta(x):
    if isinstance(x, dict):
        return x

    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return {}

    return {}

# =========================
# Chuẩn bị dữ liệu mới
# =========================

documents = []
metadatas = []
ids = []

# offset để id không bị trùng
start_idx = collection.count()

for idx, row in tqdm(df.iterrows(), total=len(df)):

    text = row.get("text", "")

    if pd.isna(text) or not str(text).strip():
        continue

    meta = parse_meta(row.get("meta", {}))

    clean_meta = {}

    for k, v in meta.items():
        if isinstance(v, (str, int, float, bool)):
            clean_meta[k] = v
        else:
            clean_meta[k] = str(v)

    documents.append(str(text))
    metadatas.append(clean_meta)

    # id mới tiếp tục từ dữ liệu cũ
    ids.append(f"doc_{start_idx + idx}")

# =========================
# Add thêm vào DB
# =========================

BATCH_SIZE = 1000

for i in range(0, len(documents), BATCH_SIZE):

    collection.add(
        documents=documents[i:i+BATCH_SIZE],
        metadatas=metadatas[i:i+BATCH_SIZE],
        ids=ids[i:i+BATCH_SIZE]
    )

print("\nĐã thêm dữ liệu mới")
print("Tổng documents:", collection.count())